# Stage 1: how well does a trained model perform on simulated data

Every model is scored on the same held-out simulated split — cell identities none of them trained
on, five localisation patterns plus a random class. Each method's embedding is read, a classifier is
fitted on it, and the same three questions are asked of all of them.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

import config
import loading
import scoring
import tables

%load_ext autoreload
%autoreload 2

## The data

In [2]:
validation = loading.simulated_validation()
validation.shape

(44706, 1)

In [3]:
validation.obs["pattern"].value_counts()

random          12724
pericellular     6442
cell-edge        6441
perinuclear      6398
intranuclear     6364
extranuclear     6337
Name: pattern, dtype: int64

In [4]:
# cell identities, and the split they come from
f"{validation.obs['cell_id'].nunique()} cells, {validation.n_obs} observations"

'64 cells, 44706 observations'

## Run

In [5]:
stage1_rows = scoring.stage1(validation)
scoring.save([stage1_rows], "stage1")
stage1_rows.shape

(17172, 22)

## Results

In [ ]:
# silhouette is computed and kept in the results file, but not shown here: the paper
# reports it over a different simulated subset, and two different numbers under one
# name invite a contradiction that is not there
results = tables.read_results()

readings = pd.concat([
    tables.mean_per_method(results, "simulated", "stage1_pattern_binary", ["kappa"]).rename(index={"kappa": "patterned vs random (kappa)"}),
    tables.mean_per_method(results, "simulated", "pattern_probe", ["kappa"]).rename(index={"kappa": "which pattern (kappa)"}),
])
readings.index.name = "reading"
tables.show(tables.label_columns(readings), "Simulated validation split", best="max")

## What the embedding should not encode

In [7]:
forest = tables.unrestricted_forest(results[results.benchmark == "negative_control"])
controls = pd.concat([
    tables.mean_per_method(forest, "simulated", "negative_control", ["kappa"], target=t).rename(index={"kappa": name})
    for t, name in [("cell_id", "cell identity"),
                    ("n_spots_interval_fixed", "transcript count"),
                    ("rotation_interval_45", "rotation")]
])
controls.index.name = "leaks"
tables.show(tables.label_columns(controls), "Negative controls (lower is better)", best="min")

**Negative controls (lower is better)**

| leaks            | SubCellSpace | Bento       | ┆ | Point-VAE  | CNN baseline | Image transformer (small) | Image transformer (large) |
|------------------|-------------:|------------:|---|-----------:|-------------:|--------------------------:|--------------------------:|
| cell identity    |       0.1650 |      0.7427 |   |     0.2745 |       0.3019 |                    0.1942 |                **0.0641** |
| transcript count |       0.2060 |      0.1401 |   | **0.1326** |       0.2517 |                    0.2222 |                    0.2005 |
| rotation         |       0.0079 | **-0.0003** |   |     0.0402 |       0.0006 |                    0.0166 |                    0.0033 |

Left of the divider: SubCellSpace against the other published tools. Right: our own alternative implementations of the same idea.